# NestedSimPy — Inventory with Lookahead Decisions

This notebook runs a NestedSimPy-specific example: a periodic-review stock whose order decision is chosen by trying each candidate quantity in inner simulations. It uses `env.decide` and `set_inner_actions`.

See the [example page](https://nestedsimpy.github.io/topical-guides/lookahead-actions.html) for the side-by-side plain/nested code.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Run the nested example

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory. In rollout mode it executes the lookahead picks, so it can differ from the plain example.

In [ ]:
%%writefile inventory_lookahead_colab.py
# --- inline prelude (replaces the examples' local _imports shim) ---
import argparse, os, random, shutil, sys, itertools
from pathlib import Path

import simpy
import nestedsimpy
from nestedsimpy import (
    NestedEnvironment, NestedResource, NestedPreemptiveResource,
    NestedStore, NestedContainer,
)
try:
    from nestedsimpy.postprocess import (
        package_latest_run, relocate_raw_artifacts, export_realizations,
    )
except Exception:  # pragma: no cover
    package_latest_run = relocate_raw_artifacts = export_realizations = None

DEFAULT_OUT_ROOT = Path("nested_output")
DEFAULT_AUTOPLOT = False
REPO_ROOT = Path(".")
PACKAGE_ROOT = Path(".")

def default_out(*parts):
    p = DEFAULT_OUT_ROOT.joinpath(*map(str, parts)); p.mkdir(parents=True, exist_ok=True); return p

def set_nested_output_folder(*parts):
    p = Path(os.path.join(*[str(x) for x in parts])); p.mkdir(parents=True, exist_ok=True); return p
# --- end prelude ---

"""
Periodic-review inventory example.

Covers:

- A periodic process with an order decision each period
- Containers: Container

Scenario:
  A single item faces Poisson demand each period. Each period opens
  with the order decision: the order-up-to rule looks at the inventory
  position (on hand plus in the pipeline) and orders the shortfall.
  The order placed one period earlier then arrives, demand realizes,
  and holding and shortage costs accrue. An order placed this period
  is on hand for the next period's demand (one period of lead time).
  This is the lookahead version: env.decide tries each candidate
  order quantity in inner simulations and executes the one with the
  lowest average cost.
"""


import numpy as np

RANDOM_SEED = 12
PERIODS = 8                # review periods
MEAN_DEMAND = 5.0          # Poisson demand per period
HOLD_COST = 1.0            # per unit on hand per period
SHORTAGE_COST = 9.0        # per unit short per period (lost sales)
ORDER_UP_TO = 10           # the rule's target position
ACTIONS = list(range(21))  # 0..20; the baseline's own order also competes
INNER_HORIZON = 4          # the lookahead window, in periods
INNER_REPS = 16            # inner branches per candidate

NESTED_OUTPUT_FOLDER = set_nested_output_folder("simpy_examples",
                                                "inventory_lookahead")


def base_policy(state):
    """Order up to ORDER_UP_TO on the inventory position."""
    position = int(state["stock"].level) + int(state["pipeline"].level)
    return max(0, ORDER_UP_TO - position)


def periods(env, state):
    while True:
        yield env.timeout(1.0)
        landing = int(state["pipeline"].level)      # last period's order, due now
        order = yield from env.decide(base_policy, state)   # this period's order
        if order > 0:
            state["pipeline"].put(order)            # arrives next period
        if landing:
            state["pipeline"].get(landing)
            state["stock"].put(landing)
        demand = int(np.random.poisson(MEAN_DEMAND))
        sales = min(int(state["stock"].level), demand)
        if sales:
            state["stock"].get(sales)
        on_hand = int(state["stock"].level)
        short = demand - sales                      # lost sales

        period_cost = HOLD_COST * on_hand + SHORTAGE_COST * short
        env.record("cost", period_cost)             # scores the branches
        state["cumulative_cost"] += period_cost


def run():
    np.random.seed(RANDOM_SEED)
    env = NestedEnvironment()
    state = {
        "stock": NestedContainer(env, capacity=float("inf"), init=10,
                                 nested_id="stock"),
        "pipeline": NestedContainer(env, capacity=float("inf"), init=0,
                                    nested_id="pipeline"),
        "cumulative_cost": 0.0,
    }
    env.process(periods(env, state))

    # No trigger configuration: NestedSimPy branches on decide's event.
    env.set_outer_stopping_condition(timeout=PERIODS + 0.5)
    env.set_inner_stopping_condition(relative_time=float(INNER_HORIZON))
    env.set_inner_repetitions(INNER_REPS)
    env.set_rng("CRN")
    env.set_outer_seed(RANDOM_SEED)
    env.set_inner_actions(ACTIONS, metric="cost", outer_run_mode="rollout")
    env.set_output_options(out_dir=NESTED_OUTPUT_FOLDER, gzip_trace=False)
    env.nested_run()
    return state["cumulative_cost"], env


if __name__ == "__main__":
    total, env = run()
    by_action = env.get_inner_results_by_action(metric="cost")
    print(f"total cost {total:.1f} over {PERIODS} periods "
          f"({len(by_action)} decisions)")
    first = min(by_action)
    for action, values in sorted(by_action[first].items(), key=lambda i: str(i[0])):
        valid = [v for v in values if v is not None]
        mean = sum(valid) / len(valid) if valid else float("nan")
        pick = " <- executed" if action == env.best_inner_action(
            trigger=first, metric="cost") else ""
        label = "base_policy" if action is None else repr(action)
        print(f"  first decision, {label:11}: mean {mean:6.1f}{pick}")


In [ ]:
# Run as a subprocess so the outer output is clean (inner branches run in separate processes).
!python inventory_lookahead_colab.py

## 3. Inspect the run

The lookahead CSVs are read back with pandas: the executed picks and the per-action score table.

In [ ]:
import glob, os
import pandas as pd

run = os.path.dirname(glob.glob("simpy_examples/inventory_lookahead/**/rollout", recursive=True)[0])

# Two of the four rollout CSVs: the executed picks, and every
# candidate's score per decision.
outer_decisions = pd.read_csv(f"{run}/rollout/outer_decisions.csv")
aggregated = pd.read_csv(f"{run}/rollout/inner_trajectories_aggregated.csv")
print(outer_decisions.to_string(index=False))  # base_policy pick = no override
print()
print(aggregated.head(8).to_string(index=False))  # base_policy row = the baseline decision, scored as one more candidate
